In [1]:
# 1. 필요한 라이브러리 설치
!pip install transformers datasets accelerate -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 126.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 97.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 108.2 MB/s eta 0:00:00


In [2]:
# 2. 파일 업로드
from google.colab import files
uploaded = files.upload()  # 여기서 두 파일 선택해서 업로드하세요 (normal, hate)


Saving call_qa_utf8_label_hate.csv to call_qa_utf8_label_hate.csv
Saving call_qa_utf8_label_normal.csv to call_qa_utf8_label_normal.csv


In [3]:
# 3. pandas로 읽기 (CP949 인코딩)
import pandas as pd

normal_file = [f for f in uploaded if "normal" in f.lower()][0]
hate_file = [f for f in uploaded if "hate" in f.lower()][0]

df_normal = pd.read_csv(normal_file, encoding="cp949")
df_hate = pd.read_csv(hate_file, encoding="cp949")

In [4]:
# 4. 라벨 부착 및 병합
df_normal["label"] = 0
df_hate["label"] = 1
df = pd.concat([df_normal, df_hate]).reset_index(drop=True)
df = df.rename(columns={"val": "text"})  # 'val' → 'text'로 변경


In [5]:
# 5. Huggingface Dataset 변환
from datasets import Dataset
dataset = Dataset.from_pandas(df)


In [6]:
# 6. Tokenizer 적용
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("monologg/koelectra-base-v3-discriminator")

def tokenize(example):
    return tokenizer(example["text"], padding="max_length", truncation=True, max_length=128)

dataset = dataset.map(tokenize)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/61.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/467 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/263k [00:00<?, ?B/s]

Map:   0%|          | 0/1677 [00:00<?, ? examples/s]

In [7]:
# 7. 훈련/평가 분할
dataset = dataset.train_test_split(test_size=0.2)


In [8]:
# 8. 모델 로딩
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "JunHwi/kold_binary", num_labels=2
)


config.json:   0%|          | 0.00/864 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/452M [00:00<?, ?B/s]

In [9]:
# 9. Training 설정
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorWithPadding

# TrainingArguments에 report_to="none" 추가
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    report_to="none",  # <- wandb 끄기
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
)


<ipython-input-9-93d525bde1bd>:20: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [10]:
# 10. 학습 시작
trainer.train()


model.safetensors:   0%|          | 0.00/452M [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss
1,0.195100,0.200214
2,0.042300,0.215948


TrainOutput(global_step=84, training_loss=0.17564475749220168, metrics={'train_runtime': 32.3814, 'train_samples_per_second': 82.825, 'train_steps_per_second': 2.594, 'total_flos': 176415962618880.0, 'train_loss': 0.17564475749220168, 'epoch': 2.0})

In [11]:
save_path = "finetuned-kold-callqa"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)


('finetuned-kold-callqa/tokenizer_config.json',
 'finetuned-kold-callqa/special_tokens_map.json',
 'finetuned-kold-callqa/vocab.txt',
 'finetuned-kold-callqa/added_tokens.json',
 'finetuned-kold-callqa/tokenizer.json')

In [12]:
import shutil
shutil.make_archive(save_path, 'zip', save_path)


'/content/finetuned-kold-callqa.zip'

In [13]:
from google.colab import files
files.download(f"{save_path}.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
from sklearn.metrics import classification_report

# 평가 데이터셋 추론
preds_output = trainer.predict(dataset["test"])
preds = preds_output.predictions.argmax(axis=1)
labels = preds_output.label_ids

# 리포트 출력
print(classification_report(labels, preds, target_names=["Normal", "Hate"]))


              precision    recall  f1-score   support

      Normal       0.87      0.98      0.92       137
        Hate       0.98      0.90      0.94       199

    accuracy                           0.93       336
   macro avg       0.93      0.94      0.93       336
weighted avg       0.94      0.93      0.93       336



In [15]:
from transformers import pipeline

pipe = pipeline("text-classification", model=trainer.model, tokenizer=tokenizer)

example = "그딴 말 할 거면 입 닫고 있어라."
print(pipe(example))
example = "오늘 날씨가 좋네요."
print(pipe(example))
example = "아니 이게 맞아요?"
print(pipe(example))
example = "도대체 왜 그러는거야 답답하게."
print(pipe(example))
example = "상담사 너 미쳤어?"
print(pipe(example))
example = "제가 배터리 좀 빌리려 하는데요"
print(pipe(example))
example = "택시 좀 신고하려고요"
print(pipe(example))

Device set to use cuda:0


[{'label': 'LABEL_1', 'score': 0.9971714615821838}]
[{'label': 'LABEL_0', 'score': 0.9941304326057434}]
[{'label': 'LABEL_1', 'score': 0.6932150721549988}]
[{'label': 'LABEL_1', 'score': 0.9971531629562378}]
[{'label': 'LABEL_1', 'score': 0.9972757697105408}]
[{'label': 'LABEL_0', 'score': 0.9760094285011292}]
[{'label': 'LABEL_0', 'score': 0.8341444730758667}]


In [16]:
# 1. Colab 파일 업로드
from google.colab import files
uploaded = files.upload()  # 사용자 파일 업로드 (예: user_sentences_labeled.csv)

# 2. pandas로 파일 읽기
import pandas as pd

file_name = list(uploaded.keys())[0]
df_eval = pd.read_csv(file_name, encoding="cp949")

# 3. 결측치 제거 및 문자열 변환
df_eval = df_eval[df_eval["sentence"].notna()]
df_eval["sentence"] = df_eval["sentence"].astype(str)

# 4. 실제 정답 라벨 준비
y_true = df_eval["label"].astype(int).tolist()

Saving user_sentences_labeled.csv to user_sentences_labeled.csv


In [17]:
import pandas as pd

# 파이프라인 구성 (fine-tuned trainer 모델과 tokenizer 사용 가정)
from transformers import pipeline

pipe = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    return_all_scores=True,
    device=-1  # GPU 환경에서는 0으로
)




Device set to use cuda:0
/usr/local/lib/python3.11/dist-packages/transformers/pipelines/text_classification.py:106: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [18]:
# 예측 수행
batch_size = 32
y_pred = []

for i in range(0, len(df_eval), batch_size):
    batch = df_eval.iloc[i:i+batch_size]["sentence"].tolist()
    outputs = pipe(batch)
    for out in outputs:
        hate_score = out[1]["score"]
        pred = 1 if hate_score >= 0.5 else 0
        y_pred.append(pred)



You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [19]:
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

# 예측 결과 평가
report_dict = classification_report(y_true, y_pred, target_names=["Normal", "Hate"], output_dict=True)
conf_matrix = confusion_matrix(y_true, y_pred)

# classification_report 보기 좋게 출력
report_df = pd.DataFrame(report_dict).T
print("🔍 [분류 성능 요약 Report]")
print(report_df.round(3))

# confusion matrix 보기 좋게 출력
conf_df = pd.DataFrame(conf_matrix,
                       index=["Actual Normal", "Actual Hate"],
                       columns=["Predicted Normal", "Predicted Hate"])

print("\n🧮 [혼동 행렬 (Confusion Matrix)]")
print(conf_df)


🔍 [분류 성능 요약 Report]
              precision  recall  f1-score   support
Normal            0.966   0.689     0.805  1851.000
Hate              0.234   0.796     0.362   221.000
accuracy          0.701   0.701     0.701     0.701
macro avg         0.600   0.743     0.583  2072.000
weighted avg      0.888   0.701     0.757  2072.000

🧮 [혼동 행렬 (Confusion Matrix)]
               Predicted Normal  Predicted Hate
Actual Normal              1276             575
Actual Hate                  45             176
